# Modelling ot section_id and tags using tf-idf and Logistic Regression

In [1]:
# import sys

# !{sys.executable} -m spacy download en_core_web_sm
# !{sys.executable} -m pip install simplemma

## Imports

In [2]:
import re
import json
import time
import string
import itertools
import joblib
from pathlib import Path

import ast
import simplemma
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.multiclass import OneVsRestClassifier

np.random.seed(42)

## Data loading and preparation

In [3]:
main_data_path = Path("./clean_data_for_section_tags_modelling")

# Load main_data
clean_df = pd.read_csv(main_data_path / "df_final_clean_filtered_new_new_v1.csv")

# Consider tags column as column where there is list in every row
clean_df["tags"] = clean_df["tags"].apply(ast.literal_eval)

# Drop unuseful columns
clean_df = clean_df.drop(columns=[
    "index", "id", "url", "published_date",
    "section_name", "tags_sections_ids", "tags_sections_names"
])

**Label encoding for section_id**

In [4]:
section_encoder = LabelEncoder()
section_label_encoding = section_encoder.fit_transform(clean_df["section_id"])
clean_df["section_id"] = section_label_encoding

**Multi-Label Binarizer for tags**

In [5]:
final_saved_tags_lst = list(sorted(set(itertools.chain.from_iterable(clean_df["tags"]))))

mlb = MultiLabelBinarizer(classes=final_saved_tags_lst)
tags_matrix = mlb.fit_transform(clean_df["tags"])

**Create additional text column**

In [6]:
def create_main_text(row):
    title = str(row['title']).strip()
    body_text = str(row['body_text']).strip()
    
    return f"{title}  {body_text}"

clean_df["main_text"] = clean_df.apply(lambda row: create_main_text(row), axis=1)

**Split into train/eval**

In [7]:
train_clean_df, test_clean_df = train_test_split(
    clean_df, 
    test_size=0.15, 
    random_state=42,
    shuffle=True,
    stratify=clean_df["section_id"]
)

Create binary tags matrices on train and test using MultiLabel-Binarizer

In [8]:
train_tags_matrix = mlb.transform(train_clean_df["tags"])
test_tags_matrix = mlb.transform(test_clean_df["tags"])

**Functions to prepare texts and use lemmatization**

In [9]:
def prepare_text(text: str):
    text = text.lower().strip()

    text = text.translate(str.maketrans("", "", string.punctuation))

    text = re.sub(r"[«»“”„“—–…®™]", "", text)

    text = re.sub(r"\d+", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [10]:
def lemmatize(text: str):
    tokens = text.split(" ")

    lemmas = [simplemma.lemmatize(w, lang="en") for w in tokens]

    return lemmas

In [11]:
start_time = time.time()

train_texts = train_clean_df["main_text"].apply(lambda x: " ".join(lemmatize(prepare_text(x))))

test_texts = test_clean_df["main_text"].apply(lambda x: " ".join(lemmatize(prepare_text(x))))

total_time = time.time() - start_time

print(f"Total time: {total_time} sec")

Total time: 41.80860900878906 sec


**Calculate Tf-Idf**

In [12]:
top_words_N = 5000  # 20000, 15000

tf_idf_vectorizer_params = {
    "encoding": "utf-8",
    "lowercase": True,
    "analyzer": "word",
    "ngram_range": (1, 2),
    "stop_words": "english",
    "max_df": 0.9,
    "min_df": 2,
    "max_features": top_words_N,
    "sublinear_tf": True,
}

tf_idf_vectorizer = TfidfVectorizer(
    **tf_idf_vectorizer_params,
)

# Get train and test tf-idf matrices with shape (n_texts, top_words_N)
train_tf_idf_matrix = tf_idf_vectorizer.fit_transform(train_texts)

test_tf_idf_matrix = tf_idf_vectorizer.transform(test_texts)

## Section_id part

## Training and evaluation for section_id

### Finding optimal hyperparameters

In [13]:
# unique_section_ids = train_clean_df["section_id"].unique().tolist()

# assert list(sorted(unique_section_ids)) == list(range(0, len(unique_section_ids)))

In [14]:
# import warnings
# warnings.filterwarnings('ignore')

# c_space = np.logspace(-3, 1, num=10)

# main_param_grid = [
#     {
#         "penalty": ["l2"],
#         "C": c_space,
#         "solver": ["lbfgs"],  # default fast solver
#     },
#     {
#         "penalty": ["l1"],
#         "C": c_space,
#         "solver": ["liblinear"],  # 'lbfgs' does not support L1
#     },
# ]

# grid_search = GridSearchCV(
#     estimator=LogisticRegression(
#         tol=1e-3,
#         fit_intercept=True,
#         class_weight="balanced",
#         random_state=42,
#         max_iter=450,
#     ),
#     param_grid=main_param_grid,
#     cv=4,
#     scoring="f1_macro",
#     verbose=3, # 1
#     n_jobs=-1,
# )

# grid_search.fit(train_tf_idf_matrix, train_clean_df["section_id"])

### Model training and making predictions

In [15]:
import warnings

warnings.filterwarnings("ignore")

best_params = {
    "C": np.float64(3.59), 
    "penalty": "l2", 
    "solver": "lbfgs",
    "tol": 1e-3,
    "fit_intercept": True,
    "class_weight": "balanced",
    "random_state": 42,
    "max_iter": 500,
}

log_reg = LogisticRegression(
    **best_params,
)

log_reg = log_reg.fit(train_tf_idf_matrix, train_clean_df["section_id"])

y_train_pred = log_reg.predict(train_tf_idf_matrix)
y_test_pred = log_reg.predict(test_tf_idf_matrix)

### Metrics computation on train and eval

In [16]:
f1_macro_train = f1_score(train_clean_df["section_id"], y_train_pred, average="macro", zero_division=0.0)
f1_macro_test = f1_score(test_clean_df["section_id"], y_test_pred, average="macro", zero_division=0.0)

precision_train = precision_score(train_clean_df["section_id"], y_train_pred, average="macro", zero_division=0.0)
precision_test = precision_score(test_clean_df["section_id"], y_test_pred, average="macro", zero_division=0.0)

recall_train = recall_score(train_clean_df["section_id"], y_train_pred, average="macro", zero_division=0.0)
recall_test = recall_score(test_clean_df["section_id"], y_test_pred, average="macro", zero_division=0.0)

print(f"[TRAIN] F1 Macro: {f1_macro_train:.3f}\n[EVAL] F1 Macro: {f1_macro_test:.3f}\n\n")
print(f"[TRAIN] Precision Macro: {precision_train:.3f}\n[EVAL] Precision Macro: {precision_test:.3f}\n\n")
print(f"[TRAIN] Recall Macro Macro: {recall_train:.3f}\n[EVAL] Recall Macro: {recall_test:.3f}")

[TRAIN] F1 Macro: 0.903
[EVAL] F1 Macro: 0.829


[TRAIN] Precision Macro: 0.900
[EVAL] Precision Macro: 0.828


[TRAIN] Recall Macro Macro: 0.906
[EVAL] Recall Macro: 0.832


We achieved good scores on eval dataset, so let's train the model on the full data (train + eval) with the same parameters, which we will use in our FastAPI app.

In [17]:
full_texts = clean_df["main_text"].apply(lambda x: " ".join(lemmatize(prepare_text(x))))

full_tf_idf_vectorizer = TfidfVectorizer(
    **tf_idf_vectorizer_params,
)

full_tf_idf_matrix = full_tf_idf_vectorizer.fit_transform(full_texts)

final_section_id_log_reg = LogisticRegression(**best_params)

final_section_id_log_reg.fit(full_tf_idf_matrix, clean_df["section_id"])

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",np.float64(3.59)
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :t

In [26]:
# # section_encoder.inverse_transform(train_clean_df["section_id"])
# joblib.dump(section_encoder, "label_encoder.joblib")
# joblib.dump(final_section_id_log_reg, "log_reg_section_id.joblib")
# joblib.dump(tf_idf_vectorizer, "tf_idf_vectorizer.joblib")

## Tags part

## Training and evaluation for tags

Separate logistic regression for each tag

In [19]:
# C_grid = np.logspace(1, 3, 20)

# best_f1_samples, best_C = None, None

# for idx, cur_C in enumerate(C_grid):
#     current_logreg = OneVsRestClassifier(
#         LogisticRegression(
#             C=cur_C,
#             penalty="l2",
#             solver="lbfgs",
#             tol=1e-3,
#             max_iter=500,
#             class_weight="balanced",
#             random_state=42,
#             fit_intercept=True,
#         ),
#         n_jobs=-1,
#     )

#     current_logreg = current_logreg.fit(train_tf_idf_matrix, train_tags_matrix)

#     current_logreg_test_pred = current_logreg.predict(test_tf_idf_matrix)

#     current_f1_samples = f1_score(test_tags_matrix, current_logreg_test_pred, average="samples", zero_division=0.0)

#     if best_f1_samples is None:
#         best_f1_samples = current_f1_samples
#         best_C = cur_C
#     else:
#         if current_f1_samples > best_f1_samples:
#             best_f1_samples = current_f1_samples
#             best_C = cur_C

#     if idx % 5 == 0:
#         print(f"best test f1_samples = {best_f1_samples}")
#         print(f"best current C: {best_C}")

In [20]:
start_time = time.time()

ovr_logreg_params = {
    "C": 30.0,
    "penalty": "l2",
    "solver": "lbfgs",
    "tol": 1e-3,
    "max_iter": 500,
    "class_weight": "balanced",
    "random_state": 42,
    "fit_intercept": True,
}

ovr_logreg = OneVsRestClassifier(
    LogisticRegression(
        **ovr_logreg_params
    ),
    n_jobs=-1,
)

ovr_logreg = ovr_logreg.fit(train_tf_idf_matrix, train_tags_matrix)

total_time_ovr = time.time() - start_time
print(f"Total time for Training OVR LogReg for Tags prediction")

chosen_threshold = 0.6

# y_train_tags_pred = ovr_logreg.predict(train_tf_idf_matrix)
# y_test_tags_pred = ovr_logreg.predict(test_tf_idf_matrix)
y_train_tags_pred = (ovr_logreg.predict_proba(train_tf_idf_matrix) > chosen_threshold).astype(int)
y_test_tags_pred = (ovr_logreg.predict_proba(test_tf_idf_matrix) > chosen_threshold).astype(int)

Total time for Training OVR LogReg for Tags prediction


In [21]:
# y_test_proba = ovr_logreg.predict_proba(test_tf_idf_matrix)

# best_threshold = 0.5
# best_test_f1 = -1

# for threshold in [0.2, 0.3, 0.4, 0.5, 0.6]:
#     test_preds = (y_test_proba > threshold).astype(int)
#     cur_f1 = f1_score(test_tags_matrix, test_preds, average="samples", zero_division=0.0)
#     if cur_f1 > best_test_f1:
#         best_test_f1 = cur_f1
#         best_threshold = threshold

# print(f"Best threshold: {best_threshold}, F1 samples: {best_test_f1:.3f}")

### Metrics computation on train and eval

In [22]:
f1_samples_tags_train = f1_score(train_tags_matrix, y_train_tags_pred, average="samples", zero_division=0.0)
f1_samples_tags_test = f1_score(test_tags_matrix, y_test_tags_pred, average="samples", zero_division=0.0)

f1_micro_tags_train = f1_score(train_tags_matrix, y_train_tags_pred, average="micro", zero_division=0.0)
f1_micro_tags_test = f1_score(test_tags_matrix, y_test_tags_pred, average="micro", zero_division=0.0)

print(f"[TRAIN] F1 Samples on Tags: {f1_samples_tags_train:.3f}\n[EVAL] F1 Samples on Tags: {f1_samples_tags_test:.3f}\n\n")
print(f"[TRAIN] F1 Micro on Tags: {f1_micro_tags_train:.3f}\n[EVAL] F1 Micro on Tags: {f1_micro_tags_test:.3f}\n\n")

[TRAIN] F1 Samples on Tags: 0.657
[EVAL] F1 Samples on Tags: 0.633


[TRAIN] F1 Micro on Tags: 0.648
[EVAL] F1 Micro on Tags: 0.622




Here we also have good scores on eval dataset and let's train the final model for tags on the full dataset

In [23]:
full_tags_matrix = mlb.transform(clean_df["tags"])

final_ovr_log_reg = OneVsRestClassifier(
    LogisticRegression(
        **ovr_logreg_params
    ),
    n_jobs=-1,
)

final_ovr_log_reg.fit(full_tf_idf_matrix, full_tags_matrix)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.","LogisticRegre...42, tol=0.001)"
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",-1
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",30.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeig

In [25]:
# joblib.dump(final_ovr_log_reg, "ovr_logreg_tags.joblib")
# joblib.dump(mlb, "mlb.joblib")

# with open("top_tags.json", "w", encoding="utf-8") as f:
#     json.dump(list(mlb.classes_), f, ensure_ascii=False)

# with open("top_sections.json", "w", encoding="utf-8") as f:
#     json.dump(list(section_encoder.classes_), f, ensure_ascii=False)